In [ ]:
# =============================================================================
# BERT自然语言推理（Natural Language Inference, NLI）
# =============================================================================
# 本节演示如何使用预训练的BERT模型进行自然语言推理任务
# NLI任务：给定一个前提（premise）和一个假设（hypothesis），判断它们的关系
# 关系类型：蕴含（entailment）、矛盾（contradiction）、中性（neutral）
#
# 例如：
# 前提："一个男人在弹吉他"
# 假设："一个男人在演奏乐器" → 蕴含（entailment）
# 假设："一个男人在睡觉" → 矛盾（contradiction）
# 假设："一个男人在公园" → 中性（neutral）

import json
import multiprocessing
import os
import time
import torch
from torch import nn
from d2l import torch as d2l
from tqdm import tqdm

print("="*50)
print("BERT自然语言推理 - 预训练模型加载与数据准备")
print("当前时间:", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
print("="*50)

# =============================================================================
# 注册预训练BERT模型
# =============================================================================
# DATA_URL是d2l库的数据服务器地址
# 我们注册两个版本的BERT：base（大）和small（小）
# base版本精度更高但计算量大，small版本适合快速实验
d2l.DATA_HUB['bert.base'] = (d2l.DATA_URL + 'bert.base.torch.zip',
                             '225d66f04cae318b841a13d32af3acc165f253ac')
d2l.DATA_HUB['bert.small'] = (d2l.DATA_URL + 'bert.small.torch.zip',
                              'c72329e68a732bef0452e4b96a1c341c8910f81f')


def load_pretrained_model(pretrained_model, num_hiddens, ffn_num_hiddens,
                          num_heads, num_layers, dropout, max_len, devices):
    """
    加载预训练BERT模型
    
    参数:
        pretrained_model: 模型名称（'bert.base'或'bert.small'）
        num_hiddens: 隐藏层维度
        ffn_num_hiddens: 前馈网络隐藏层维度
        num_heads: 注意力头数
        num_layers: Transformer层数
        dropout: Dropout概率
        max_len: 最大序列长度
        devices: 计算设备列表
    
    返回:
        bert: 加载好预训练权重的BERT模型
        vocab: 词汇表对象
    """
    print(f"\n[1/5] 开始加载预训练模型: {pretrained_model}")
    start_time = time.time()
    
    print(" - 下载并解压模型...")
    # download_extract会自动处理下载、缓存和解压
    data_dir = d2l.download_extract(pretrained_model)
    
    print(" - 加载词汇表...")
    # BERT使用WordPiece分词，词汇表包含约3万-6万个token
    vocab = d2l.Vocab()
    vocab_path = os.path.join(data_dir, 'vocab.json')
    # 从JSON文件加载词汇表映射
    vocab.idx_to_token = json.load(open(vocab_path))
    vocab.token_to_idx = {token: idx for idx, token in enumerate(vocab.idx_to_token)}
    print(f"   词汇表大小: {len(vocab)} tokens")
    
    print(" - 初始化BERT模型架构...")
    # 创建BERT模型实例
    # 参数需要与预训练时使用的配置一致
    bert = d2l.BERTModel(
        len(vocab),          # 词汇表大小
        num_hiddens,         # 隐藏层维度（base:768, small:256）
        norm_shape=[256],    # LayerNorm的形状
        ffn_num_input=256,   # 前馈网络输入维度
        ffn_num_hiddens=ffn_num_hiddens,  # 前馈网络隐藏层（通常4倍隐藏层）
        num_heads=4,         # 注意力头数
        num_layers=2,        # Transformer层数
        dropout=0.2,         # Dropout率
        max_len=max_len,     # 最大序列长度
        key_size=256,        # 注意力key维度
        query_size=256,      # 注意力query维度
        value_size=256,      # 注意力value维度
        hid_in_features=256, # 隐藏层输入特征数
        mlm_in_features=256, # MLM输入特征数
        nsp_in_features=256  # NSP输入特征数
    )
    
    print(" - 加载预训练权重...")
    # 加载预训练的参数权重
    # 这些权重是通过在大规模语料（Wiki+Books）上预训练得到的
    params_path = os.path.join(data_dir, 'pretrained.params')
    bert.load_state_dict(torch.load(params_path))
    
    elapsed = time.time() - start_time
    print(f"✓ 预训练模型加载完成! 耗时: {elapsed:.2f}秒")
    return bert, vocab


print("\n[阶段1] 准备加载预训练BERT模型")
# 检测可用设备（GPU优先）
devices = d2l.try_all_gpus()
print(f"检测到可用设备: {devices}")

# 加载bert.small版本（较小的模型，适合本地实验）
# 如需更高精度可改用'bert.base'
bert, vocab = load_pretrained_model(
    'bert.small', num_hiddens=256, ffn_num_hiddens=512, num_heads=4,
    num_layers=2, dropout=0.1, max_len=512, devices=devices)

In [ ]:
# 微调BERT
class BERTClassifier(nn.Module):
    def __init__(self, bert):
        super(BERTClassifier, self).__init__()
        self.encoder = bert.encoder
        self.hidden = bert.hidden
        self.output = nn.Linear(256, 3)

    def forward(self, inputs):
        tokens_X, segments_X, valid_lens_x = inputs
        encoded_X = self.encoder(tokens_X, segments_X, valid_lens_x)
        return self.output(self.hidden(encoded_X[:, 0, :]))
    
net = BERTClassifier(bert)

lr, num_epochs = 1e-4, 5
trainer = torch.optim.Adam(net.parameters(), lr=lr)
loss = nn.CrossEntropyLoss(reduction='none')
d2l.train_ch13(net, train_iter, test_iter, loss, trainer, num_epochs,
    devices)

In [3]:
# 微调BERT模型
import time
from tqdm import tqdm
import torch
from torch import nn
from d2l import torch as d2l

print("\n" + "="*50)
print("开始微调BERT模型")
print(f"当前时间: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())}")
print("="*50)

class BERTClassifier(nn.Module):
    def __init__(self, bert):
        super(BERTClassifier, self).__init__()
        print("\n[1/5] 初始化BERT分类器")
        print(" - 使用预训练BERT的编码器")
        self.encoder = bert.encoder
        print(" - 使用预训练BERT的隐藏层")
        self.hidden = bert.hidden
        print(" - 添加输出层 (256 -> 3)")
        self.output = nn.Linear(256, 3)
        
        # 打印模型结构
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"模型总参数: {total_params:,}")
        print(f"可训练参数: {trainable_params:,} ({trainable_params/total_params:.2%})")
        
        # 设备分配
        self.device = next(self.parameters()).device
        print(f"模型已分配到设备: {self.device}")

    def forward(self, inputs):
        tokens_X, segments_X, valid_lens_x = inputs
        
        # 打印输入形状用于调试
        if hasattr(self, 'debug') and self.debug:
            print(f"\n[输入调试]")
            print(f"tokens_X: {tokens_X.shape} (设备: {tokens_X.device})")
            print(f"segments_X: {segments_X.shape} (设备: {segments_X.device})")
            print(f"valid_lens_x: {valid_lens_x.shape} (设备: {valid_lens_x.device})")
        
        # 通过BERT编码器
        encoded_X = self.encoder(tokens_X, segments_X, valid_lens_x)
        
        # 提取[CLS]标记的表示
        cls_representation = encoded_X[:, 0, :]
        
        # 通过隐藏层和输出层
        hidden_output = self.hidden(cls_representation)
        output = self.output(hidden_output)
        
        return output

# 启用调试模式 (设置为True以打印详细输入信息)
DEBUG_MODE = False

print("\n[2/5] 创建BERT分类器")
net = BERTClassifier(bert)
if DEBUG_MODE:
    print(" - 启用调试模式")
    net.debug = True

# 将模型移动到GPU
print("\n[3/5] 将模型分配到设备")
net = net.to(devices[0])
print(f"模型已移动到: {next(net.parameters()).device}")

# 设置训练参数
lr, num_epochs = 1e-4, 5
print(f"\n[4/5] 设置训练参数")
print(f"学习率: {lr}")
print(f"训练周期: {num_epochs}")
print(f"批大小: {batch_size}")
print(f"训练设备: {devices}")

# 创建优化器和损失函数
trainer = torch.optim.Adam(net.parameters(), lr=lr)
loss = nn.CrossEntropyLoss(reduction='none')

print("\n[5/5] 开始训练过程")
print("="*50)

# 自定义训练函数以添加进度监控
def train_bert_with_progress(net, train_iter, test_iter, loss, trainer, num_epochs, devices):
    # 记录历史指标
    train_loss_history = []
    train_acc_history = []
    test_acc_history = []
    epoch_times = []
    
    # 主训练循环
    for epoch in range(num_epochs):
        print(f"\n{'='*60}")
        print(f"Epoch {epoch+1}/{num_epochs}")
        print(f"开始时间: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())}")
        print('-'*60)
        
        # 训练模式
        net.train()
        metric = d2l.Accumulator(3)  # 训练损失总和, 训练准确率总和, 样本数
        
        # 创建训练进度条
        train_bar = tqdm(
            enumerate(train_iter), 
            total=len(train_iter),
            desc=f"训练 Epoch {epoch+1}",
            ncols=100,
            unit='batch'
        )
        
        start_time = time.time()
        
        for i, batch in train_bar:
            # 准备数据
            inputs, labels = batch
            inputs = [x.to(devices[0]) for x in inputs]
            labels = labels.to(devices[0])
            
            # 前向传播
            trainer.zero_grad()
            outputs = net(inputs)
            
            # 计算损失
            l = loss(outputs, labels)
            
            # 反向传播
            l.mean().backward()
            trainer.step()
            
            # 更新指标
            with torch.no_grad():
                metric.add(
                    float(l.sum()), 
                    float(d2l.accuracy(outputs, labels)), 
                    labels.numel()
                )
                
            # 更新进度条
            avg_loss = metric[0] / metric[2]
            avg_acc = metric[1] / metric[2]
            train_bar.set_postfix(
                loss=f"{avg_loss:.4f}", 
                acc=f"{avg_acc:.4f}",
                lr=f"{lr:.0e}"
            )
        
        # 计算训练指标
        train_loss = metric[0] / metric[2]
        train_acc = metric[1] / metric[2]
        train_loss_history.append(train_loss)
        train_acc_history.append(train_acc)
        
        # 评估模式
        net.eval()
        test_acc = d2l.evaluate_accuracy_gpu(net, test_iter)
        test_acc_history.append(test_acc)
        
        # 计算时间
        epoch_time = time.time() - start_time
        epoch_times.append(epoch_time)
        
        # 打印epoch结果
        print(f"\nEpoch {epoch+1} 结果:")
        print(f"  训练损失: {train_loss:.4f}")
        print(f"  训练准确率: {train_acc:.4f}")
        print(f"  测试准确率: {test_acc:.4f}")
        print(f"  耗时: {epoch_time:.2f}秒 ({epoch_time/60:.2f}分钟)")
        print(f"  平均速度: {metric[2]/epoch_time:.2f} 样本/秒")
        
        # 内存使用情况
        if torch.cuda.is_available():
            max_mem = torch.cuda.max_memory_allocated(devices[0]) / (1024**3)  # GB
            cur_mem = torch.cuda.memory_allocated(devices[0]) / (1024**3)  # GB
            print(f"  GPU内存使用: {cur_mem:.2f} GB (峰值: {max_mem:.2f} GB)")
    
    # 训练结束统计
    total_time = sum(epoch_times)
    avg_epoch_time = total_time / num_epochs
    
    print("\n" + "="*60)
    print(f"训练完成! 总耗时: {total_time:.2f}秒 ({total_time/60:.2f}分钟)")
    print(f"平均epoch时间: {avg_epoch_time:.2f}秒")
    print(f"最终训练准确率: {train_acc_history[-1]:.4f}")
    print(f"最终测试准确率: {test_acc_history[-1]:.4f}")
    
    # 返回历史记录
    return train_loss_history, train_acc_history, test_acc_history

# 开始训练
print(f"训练样本数: {len(train_set)}")
print(f"测试样本数: {len(test_set)}")
print(f"训练批次: {len(train_iter)}")
print(f"测试批次: {len(test_iter)}")
print("="*50)

# 调用自定义训练函数
train_loss_hist, train_acc_hist, test_acc_hist = train_bert_with_progress(
    net, train_iter, test_iter, loss, trainer, num_epochs, devices)

print("\n" + "="*50)
print("BERT微调完成!")
print(f"结束时间: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())}")

print("="*50)


开始微调BERT模型
当前时间: 2025-06-03 14:34:03

[2/5] 创建BERT分类器

[1/5] 初始化BERT分类器
 - 使用预训练BERT的编码器
 - 使用预训练BERT的隐藏层
 - 添加输出层 (256 -> 3)
模型总参数: 16,613,635
可训练参数: 16,613,635 (100.00%)
模型已分配到设备: cpu

[3/5] 将模型分配到设备
模型已移动到: cpu

[4/5] 设置训练参数
学习率: 0.0001
训练周期: 5
批大小: 512
训练设备: [device(type='cpu')]

[5/5] 开始训练过程
训练样本数: 549367
测试样本数: 9824
训练批次: 1073
测试批次: 20

Epoch 1/5
开始时间: 2025-06-03 14:34:04
------------------------------------------------------------


训练 Epoch 1: 100%|█████| 1073/1073 [1:37:14<00:00,  5.44s/batch, acc=0.6396, loss=0.7990, lr=1e-04]



Epoch 1 结果:
  训练损失: 0.7990
  训练准确率: 0.6396
  测试准确率: 0.7139
  耗时: 5869.44秒 (97.82分钟)
  平均速度: 93.60 样本/秒

Epoch 2/5
开始时间: 2025-06-03 16:11:53
------------------------------------------------------------


训练 Epoch 2: 100%|█████| 1073/1073 [1:37:12<00:00,  5.44s/batch, acc=0.7231, loss=0.6586, lr=1e-04]



Epoch 2 结果:
  训练损失: 0.6586
  训练准确率: 0.7231
  测试准确率: 0.7475
  耗时: 5865.23秒 (97.75分钟)
  平均速度: 93.66 样本/秒

Epoch 3/5
开始时间: 2025-06-03 17:49:38
------------------------------------------------------------


...


Epoch 3 结果:
  训练损失: 0.5973
  训练准确率: 0.7536
  测试准确率: 0.7673
  耗时: 5845.65秒 (97.43分钟)
  平均速度: 93.98 样本/秒

Epoch 4/5
开始时间: 2025-06-03 19:27:04
------------------------------------------------------------


训练 Epoch 4: 100%|█████| 1073/1073 [1:35:15<00:00,  5.33s/batch, acc=0.7748, loss=0.5543, lr=1e-04]



Epoch 4 结果:
  训练损失: 0.5543
  训练准确率: 0.7748
  测试准确率: 0.7746
  耗时: 5748.94秒 (95.82分钟)
  平均速度: 95.56 样本/秒

Epoch 5/5
开始时间: 2025-06-03 21:02:53
------------------------------------------------------------


训练 Epoch 5: 100%|█████| 1073/1073 [1:35:13<00:00,  5.32s/batch, acc=0.7899, loss=0.5216, lr=1e-04]



Epoch 5 结果:
  训练损失: 0.5216
  训练准确率: 0.7899
  测试准确率: 0.7752
  耗时: 5745.20秒 (95.75分钟)
  平均速度: 95.62 样本/秒

训练完成! 总耗时: 29074.47秒 (484.57分钟)
平均epoch时间: 5814.89秒
最终训练准确率: 0.7899
最终测试准确率: 0.7752

BERT微调完成!
结束时间: 2025-06-03 22:38:38


In [1]:
import torch
print(torch.__version__)
print(f"CUDA可用: {torch.cuda.is_available()}")
print(f"检测到的CUDA设备数量: {torch.cuda.device_count()}")
print(torch.version.cuda)

2.5.1
CUDA可用: False
检测到的CUDA设备数量: 0
None
